# Zero-VRAM Dual-Artifact Ingestion: Semantic Chunking & Cluster Analysis

This notebook provides mathematical validation and exploratory cluster analysis for Phase 4 ingestion:
1. **Semantic Markdown Chunking**: Preserves header boundaries (, , ).
2. **Chunk Size Distribution**: Analyzes token/character bounds to verify prompt fits.
3. **FastEmbed CPU Vectorization**: Projects 384-dimensional dense vectors into 2D via PCA to confirm semantic clustering.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Path resolution
REPO_ROOT = Path("..").resolve() if Path("..").joinpath("scripts").exists() else Path(".").resolve()
sys.path.insert(0, str(REPO_ROOT))

from scripts.ingestion_core import DocumentProcessor
from fastembed import TextEmbedding

print("✅ Ingestion core and analytical dependencies loaded successfully.")


In [ ]:
# Select target documentation file for analysis
target_doc = REPO_ROOT / "docs" / "architecture_guide.md"
if not target_doc.exists():
    target_doc = REPO_ROOT / "README.md"

processor = DocumentProcessor(chunk_size=800, chunk_overlap=80)
chunks = processor.process_file(target_doc)

print(f"📄 Target Document: {target_doc.name}")
print(f"📊 Total Semantic Chunks: {len(chunks)}")
if chunks:
    print(f"Sample Chunk 0 Header: {chunks[0]["metadata"].get("Header 1", "N/A")} -> {chunks[0]["metadata"].get("Header 2", "N/A")}")
    print(f"Sample Deterministic ID: {chunks[0]["id"]}")


In [ ]:
# 1. Chunk Character Length Distribution
char_lengths = [c["metadata"]["char_length"] for c in chunks]

plt.figure(figsize=(9, 4.5), dpi=100)
plt.hist(char_lengths, bins=10, color="#1f77b4", edgecolor="black", alpha=0.8)
plt.axvline(np.mean(char_lengths), color="crimson", linestyle="--", linewidth=1.5, label=f"Mean ({np.mean(char_lengths):.1f} chars)")
plt.axvline(processor.chunk_size, color="darkorange", linestyle=":", linewidth=1.5, label=f"Ceiling ({processor.chunk_size} chars)")

plt.title("Semantic Chunk Character Length Distribution", fontsize=12, fontweight="bold")
plt.xlabel("Character Count", fontsize=10)
plt.ylabel("Number of Chunks", fontsize=10)
plt.grid(axis="y", linestyle=":", alpha=0.6)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

print(f"Summary: Mean={np.mean(char_lengths):.1f} | Min={min(char_lengths)} | Max={max(char_lengths)} | Std={np.std(char_lengths):.1f}")


In [ ]:
# 2. FastEmbed Dense Vectorization & PCA 2D Cluster Analysis
embedding_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5", threads=4)
texts = [c["text"] for c in chunks]
embeddings = np.array(list(embedding_model.embed(texts)))

print(f"Dense Vector Matrix: {embeddings.shape} (Dim: 384)")

# Project to 2D via PCA
n_components = min(2, len(chunks))
pca = PCA(n_components=n_components, random_state=42)
reduced_vectors = pca.fit_transform(embeddings)

categories = [
    c["metadata"].get("Header 2") or c["metadata"].get("Header 1") or "General"
    for c in chunks
]
unique_cats = list(dict.fromkeys(categories))

plt.figure(figsize=(9, 6), dpi=100)
for idx, cat in enumerate(unique_cats):
    mask = [c == cat for c in categories]
    x_coords = reduced_vectors[mask, 0] if reduced_vectors.shape[1] > 0 else np.zeros(sum(mask))
    y_coords = reduced_vectors[mask, 1] if reduced_vectors.shape[1] > 1 else np.zeros(sum(mask))
    plt.scatter(x_coords, y_coords, label=cat[:30], s=100, alpha=0.85, edgecolor="black", linewidth=0.5)
    for i, (x, y) in enumerate(zip(x_coords, y_coords)):
        plt.annotate(f"C{i+1}", (x, y), textcoords="offset points", xytext=(5, 5), fontsize=8, alpha=0.8)

var_ratio = getattr(pca, "explained_variance_ratio_", [0, 0])
title_str = "Semantic Embedding Cluster Projection (PCA 2D) - Explained Variance: " + str(round(sum(var_ratio)*100, 1)) + "%"
plt.title(title_str, fontsize=12, fontweight="bold")
plt.xlabel("Principal Component 1", fontsize=10)
plt.ylabel("Principal Component 2", fontsize=10)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", title="Markdown Header Category", fontsize=9)
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()
print("✅ Visual cluster analysis rendered successfully without exceptions.")
